# E-Commerce Fulfilment Analytics Take-Home Task

In [5]:
import pandas as pd
from pathlib import Path

## Data Extraction

In [6]:
csv_path1 = Path.home() /"downloads"/"shipments.csv"
csv_path2 = Path.home() /"downloads"/"orders.csv"
csv_path3 = Path.home() /"downloads"/"refunds.csv"

shipments_df = pd.read_csv(csv_path1, low_memory = False)
orders_df = pd.read_csv(csv_path2, low_memory = False)
refunds_df = pd.read_csv(csv_path3, low_memory = False)

In [16]:
shipments_df.head()

,shipment_id,order_id,courier,ship_date,delivered_date,status,shipping_days
0,1,1,Arrow,2025-02-23,2025-02-27,Delivered,4.0
1,2,2,Arrow,2025-01-17,2025-01-19,Delivered,2.0
2,3,3,Cyclone,2025-03-14,2025-03-15,Delivered,1.0
3,4,4,Cyclone,2025-03-02,2025-03-04,Delivered,2.0
4,5,4,Cyclone,2025-03-04,2025-03-05,Delivered,1.0


In [8]:
orders_df.head()

,order_id,customer_id,order_date,region,category,items,order_value
0,1,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-02-21,South,Home,2,79.43
1,2,23b8c1e9-3924-46de-beb1-3b9046685257,2025-01-15,North,Clothing,4,35.45
2,3,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,2025-03-13,West,Sports,3,27.30
3,4,972a8469-1641-4f82-8b9d-2434e465e150,2025-03-02,North,Electronics,2,103.98
4,5,17fc695a-07a0-4a6e-8822-e8f36c031199,2025-01-21,East,Clothing,2,50.75


In [9]:
refunds_df.head()

,refund_id,order_id,refund_date,refund_reason,refund_amount
0,1,5,2025-02-12,Late,41.89
1,2,21,2025-03-06,Damaged,63.19
2,3,86,2025-03-24,Damaged,46.11
3,4,88,2025-04-10,Late,90.17
4,5,89,2025-04-09,Wrong Item,12.99


### Section 1

In [72]:
couriers = pd.Series.value_counts(shipments_df['courier'])
couriers

courier
Bolt       1924
Cyclone    1920
Arrow      1901
Name: count, dtype: int64

In [21]:
pd.Series.mean(shipments_df['shipping_days'])

2.5958587539286375

In [23]:
merged_df = shipments_df.merge(orders_df, on="order_id")
merged_df.head()

,shipment_id,order_id,courier,ship_date,delivered_date,status,shipping_days,customer_id,order_date,region,category,items,order_value
0,1,1,Arrow,2025-02-23,2025-02-27,Delivered,4.0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2025-02-21,South,Home,2,79.43
1,2,2,Arrow,2025-01-17,2025-01-19,Delivered,2.0,23b8c1e9-3924-46de-beb1-3b9046685257,2025-01-15,North,Clothing,4,35.45
2,3,3,Cyclone,2025-03-14,2025-03-15,Delivered,1.0,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,2025-03-13,West,Sports,3,27.30
3,4,4,Cyclone,2025-03-02,2025-03-04,Delivered,2.0,972a8469-1641-4f82-8b9d-2434e465e150,2025-03-02,North,Electronics,2,103.98
4,5,4,Cyclone,2025-03-04,2025-03-05,Delivered,1.0,972a8469-1641-4f82-8b9d-2434e465e150,2025-03-02,North,Electronics,2,103.98


In [28]:
avg_shipping = merged_df.groupby("region")["shipping_days"].mean()
avg_shipping

region
East     2.581818
North    2.622781
South    2.612058
West     2.555251
Name: shipping_days, dtype: float64

In [30]:
product_order_val = merged_df.groupby("category")["order_value"].sum()
product_order_val

category
Clothing        68969.73
Electronics    175540.07
Home           119112.77
Sports          56200.67
Toys            33621.70
Name: order_value, dtype: float64

In [64]:
merged_df2 = orders_df.merge(refunds_df)
refunds = pd.Series.value_counts(merged_df2['category'])
orders = pd.Series.value_counts(merged_df['category'])
refunds/orders

category
Clothing       0.079476
Electronics    0.071429
Home           0.061321
Sports         0.063354
Toys           0.076100
Name: count, dtype: float64

In [67]:
from statsmodels.stats.proportion import proportions_ztest

counts = [refunds['Clothing'], refunds['Home']]
nobs = [orders['Clothing'], orders['Home']]

stat, p = proportions_ztest(counts, nobs)
p

0.0690192698344991

In [73]:
merged_df3 = shipments_df.merge(refunds_df, on = "order_id")
merged_df3
returned_refunded = merged_df3[merged_df3["status"] == "Returned"].groupby("courier").size()
returned_refunded/couriers

courier
Arrow      0.001052
Bolt       0.003119
Cyclone    0.003125
dtype: float64

In [133]:
merged_df4 = merged_df.merge(refunds_df, on="order_id", how="left")
merged_df4["refund_amount"] = merged_df4["refund_amount"].fillna(0)
merged_df4["Net_revenue"] = merged_df4['order_value'] - merged_df4['refund_amount']
total_parcels = merged_df4.groupby(['courier','category']).size()
net_rev = merged_df4.groupby(['courier','category'])["Net_revenue"].sum()
net_rev/total_parcels

courier  category   
Arrow    Clothing        55.822897
         Electronics    113.203607
         Home            76.756826
         Sports          66.145117
         Toys            37.345693
Bolt     Clothing        56.624560
         Electronics    111.945227
         Home            76.000397
         Sports          65.173298
         Toys            36.377887
Cyclone  Clothing        56.217127
         Electronics    111.829076
         Home            76.323233
         Sports          66.606932
         Toys            38.066667
dtype: float64